In [ ]:
import pandas as pd
import logomaker
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

import seaborn as sns

In [ ]:
# get feature importances based on SHAP
x = 20
feat_imp_df = pd.read_csv('RF_results/raw/cluster_1_Feature_importance.tsv', sep='\t', usecols=[0,2],names=['col_name','feature_importance_vals'],header=0)
top_50_ft_df = feat_imp_df.head(n=x).set_index('col_name').T
print(top_50_ft_df)

In [ ]:
feat_imp_df

In [ ]:
# plot shap importance across the sequence and a rolling average to see the peaks 

feat_imp_df[['position','amino_acid']] = feat_imp_df['col_name'].str.split('_',expand=True)
feat_imp_df['position'] = feat_imp_df['position'].astype('int64')
feat_imp_df.sort_values('position',inplace=True)

# feat_imp_max_df = feat_imp_df.groupby('position')['feature_importance_vals'].max()
# feat_imp_max_df

feat_imp_sum_df = feat_imp_df.groupby('position')['feature_importance_vals'].sum().reset_index()
feat_imp_sum_df
feat_imp_sum_df[ 'rolling_avg' ] = feat_imp_sum_df.feature_importance_vals.rolling(5).mean() 
feat_imp_sum_df

sns.set_context('paper')
figure(figsize=(300, 30), dpi=96)
sns.lineplot(x=feat_imp_sum_df.position, y=feat_imp_sum_df['rolling_avg'], linewidth = 25, color='darkmagenta')
sns.scatterplot(x=feat_imp_sum_df.position, y=feat_imp_sum_df['feature_importance_vals'], s=5000, hue=feat_imp_sum_df['feature_importance_vals'], alpha=0.7)

#ax = plt.gca()

# Add labels to your graph
plt.xlabel('MSA Position')
plt.ylabel('Sum SHAP feature importance value')
plt.title("SHAP values across sequence position")
# plt.xticks(x[0::3])
plt.legend()

# for label in ax.xaxis.get_ticklabels():
#     if ax.xaxis.get_ticklabels().index(label)%50:
#         continue
#     label.set_visible(False)

#ax.set_xticks(range(0, 500))
#ax.set_xticklabels(feat_imp_sum_df.position, rotation=90, fontsize=80)
plt.xticks(range(0,700,50),fontsize=100, rotation=90)
plt.yticks(fontsize=80)
plt.show()


In [ ]:
data_df_merged_balanced = pd.read_csv('RF_results/raw/cluster_1_data_df.tsv', sep='\t')
data_df_merged_balanced['biome'] = data_df_merged_balanced['biome'].str.replace(":",";")
data_df_merged_balanced

In [ ]:
data_df_merged_balanced.groupby("ecosystem_subtype").sum()

In [ ]:
# merge with the bigger dataframe with biomes
data_df_merged_balanced_1 = data_df_merged_balanced.groupby('ecosystem_subtype').apply(lambda x: (x==1).sum()).T
data_df_merged_balanced_1
data_df_merged_balanced_1['proportion_lake'] = data_df_merged_balanced_1['Lake']/data_df_merged_balanced.groupby('ecosystem_subtype').count().T.Lake*100*-1
data_df_merged_balanced_1['proportion_oceanic'] = data_df_merged_balanced_1['Oceanic']/data_df_merged_balanced.groupby('ecosystem_subtype').count().T.Oceanic*100

print(data_df_merged_balanced_1)
data_df_merged_balanced_1 = data_df_merged_balanced_1.T


In [ ]:
top_200_features = data_df_merged_balanced_1[top_50_ft_df.columns]
top_200_features
print(top_200_features.T.head(n=20))

# only proportions
top_200_features.drop(labels=['Lake','Oceanic'], inplace=True)
top_200_features

#### Sorting values for the plot

In [ ]:
# ORIGINAL BY FEATURE IMPORTANCE (most important features shown at the top)
top_200_features.reset_index(inplace=True)
# print(top_200_features)
# melt
# top_200_features_melted = pd.melt(top_200_features, id_vars=['ecosystem_subtype'],ignore_index=False)
# top_200_features_melted


# LARGEST DIFFERENCE IN THE LOG2 FOLD CHANGE BETWEEN FRESHWATER AND MARINE
# top_200_features = top_200_features.sort_values(axis=1, by='diff', ascending=False)     #.drop(index=('diff'),inplace=True)
# top_200_features=top_200_features.loc[['Freshwater','Marine']]
# top_200_features.reset_index(inplace=True)
# print(top_200_features)
# # melt
# top_200_features_melted = pd.melt(top_200_features, id_vars=['ecosystem_type'],ignore_index=False)
# top_200_features_melted


# FEATURE POSITION
# top_200_features_1 = top_200_features.drop(index=('diff'))
# top_200_features_1.reset_index(inplace=True)
top_200_features_melted = pd.melt(top_200_features, id_vars=['ecosystem_subtype'])     # melt data
top_200_features_melted[['0','position','AA','1']]= top_200_features_melted['variable'].str.split('(\d+)_([A-Za-z]+|-)', expand=True)   #split the feature into position and the AA
top_200_features_melted[['position']]=top_200_features_melted[['position']].apply(pd.to_numeric)        # covert the position to a numeric
top_200_features_melted = top_200_features_melted.drop(['0','1'],axis=1).sort_values("position")     # sort by position


# BY BIOME (KINDA)
# top_200_features.drop(index=('diff'),inplace=True)
# top_200_features.reset_index(inplace=True)
# top_200_features_melted = pd.melt(top_200_features, id_vars=['ecosystem_type'])     # melt data
# top_200_features_melted = top_200_features_melted.sort_values("value")

In [ ]:
top_200_features_melted

In [ ]:
print(top_200_features_melted.head(40))

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

sns.set_theme(style="whitegrid")
my_palette = sns.diverging_palette(h_neg=100, h_pos=200,s=100,l=60, n=2)
min,max = 2, 50
plt.figure(figsize=(6,x/5))

b = sns.barplot(
    data=top_200_features_melted,
    x="value",
    y="variable",
    hue="ecosystem_subtype",
    palette=['#2B8CBF','#9dcc7e'],
    dodge=False,
    width=-.5,  
)

# show the graph
# b.set_yticklabels(top_200_features_melted.variable)
b.set_xlabel("% feature appears in lake/oceanic sequences",fontsize=15)
b.set_ylabel(f"Top {x} Features",fontsize=15)
plt.show()

## Create a sequence logo of the top (x) positions

In [ ]:
# sequence logo creation
import logomaker

top_200_features_1 = top_200_features
shap_importance = feat_imp_df
top_200_features_melted = pd.melt(top_200_features_1, id_vars=['ecosystem_subtype'])     # melt data
top_200_features_melted[['0','position','AA','1']]= top_200_features_melted['variable'].str.split('(\d+)_([A-Za-z]+|-)', expand=True)   #split the feature into position and the AA
top_200_features_melted[['position']]=top_200_features_melted[['position']].apply(pd.to_numeric)        # covert the position to a numeric
top_200_features_melted = top_200_features_melted.drop(['0','1'],axis=1).sort_values("position")     # sort by position


# adding shap or gini importance to dataframe
top_200_features_melted.reset_index(inplace=True, names='importance_order')
xx = top_200_features_melted.merge(shap_importance, left_on='variable', right_on='col_name')
top_200_features_melted = xx.drop('col_name', axis=1)
print(top_200_features_melted)

position_df = top_200_features_melted[['value', 'position_x', 'AA','ecosystem_subtype']]
print(position_df)

position_df_marine = position_df.loc[position_df['ecosystem_subtype'] == 'proportion_oceanic']
print(position_df_marine)
position_df_new_marine = position_df_marine.pivot_table(index=["position_x"], columns='AA', values='value', fill_value=0.00001)
# position_df_new_marine.drop('-',inplace=True,axis=1)


position_df_freshwater = position_df.loc[position_df['ecosystem_subtype'] == 'proportion_lake']
position_df_new_freshwater = position_df_freshwater.pivot_table(index=["position_x"], columns='AA', values='value', fill_value=0.00001)
print(position_df_new_freshwater)
# position_df_new_freshwater.drop('-',inplace=True, axis=1)

position_df_new_marine.index = pd.to_numeric(position_df_new_marine.index)
position_df_new_freshwater.index = pd.to_numeric(position_df_new_freshwater.index)

In [ ]:
position_df_new_marine

In [ ]:
top_200_features_melted

In [ ]:
# top_200_features_melted
top_200_features_melted_for_bars = top_200_features_melted.sort_values(
    by=['variable', 'value'], 
    ascending=[True, False], 
    key=lambda x: x if x.name == 'variable' else abs(x)  # Only apply abs to 'value' column
).groupby('variable').head(n=1)

# top_200_features_melted_for_bars.loc[top_200_features_melted_for_bars['ecosystem_subtype'] == 'proportion_freshwater']
top_200_features_melted_for_bars

In [ ]:
top_200_features_melted_for_bars.loc[top_200_features_melted_for_bars['ecosystem_subtype'] == 'proportion_lake']


In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

top_200_features_melted["alpha"] = (top_200_features_melted["feature_importance_vals"] - top_200_features_melted["feature_importance_vals"].min()) / (top_200_features_melted["feature_importance_vals"].max() - top_200_features_melted["feature_importance_vals"].min())
top_200_features_melted["alpha"] = 0.2 + 0.8 * top_200_features_melted["alpha"]

sns.set_theme(style="whitegrid")
my_palette = sns.diverging_palette(h_neg=100, h_pos=200,s=100,l=60, n=2)
min,max = 2, 50
plt.figure(figsize=(x/5,6))

b = sns.barplot(
    data=top_200_features_melted,
    x="position_x",
    y="value",
    hue="ecosystem_subtype",
    palette=['#2B8CBF','#9dcc7e'],
    dodge=False,
    width=-.5,
)
# Apply alpha per bar
for patch, alpha in zip(b.patches, top_200_features_melted["alpha"]):
    patch.set_alpha(alpha)

# show the graph
# b.set_yticklabels(top_200_features_melted.variable)
b.set_xlabel("% feature appears in lake/oceanic sequences",fontsize=15)
b.set_ylabel(f"Top {x} Features",fontsize=15)
plt.xticks(fontsize=10, rotation=90)
plt.show()

In [ ]:
position_df_new_marine

In [ ]:
# Create the new figure and subplots
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(30, 5), dpi = 120)

# Create the marine logo directly on the first subplot
AA_logo_marine = logomaker.Logo(
    position_df_new_marine,
    ax=axes[1],  # Use the first subplot's axes
    font_name='Arial Rounded MT Bold',
    color_scheme='dmslogo_charge',
    stack_order='small_on_top',  # Ensures visibility of larger letters
    width=2.5
)

# Create the freshwater logo directly on the second subplot
AA_logo_fresh = logomaker.Logo(
    position_df_new_freshwater,
    ax=axes[2],  # Use the second subplot's axes
    font_name='Arial Rounded MT Bold',
    color_scheme='dmslogo_charge',
    flip_below=False,
    baseline_width = 10,
    stack_order='small_on_top',  # Ensures visibility of larger letters
    width=2.5

)

# create marine bars above first seqlogo (axes[0])
data_m=top_200_features_melted_for_bars.loc[top_200_features_melted_for_bars['ecosystem_subtype'] == 'proportion_oceanic']
data_m["position_x"] = data_m["position_x"].astype(float)

axes[0].bar(
    data_m["position_x"], 
    data_m["feature_importance_vals"], 
    color="#9dcc7e",
    width=2  # Adjust width to match sequence logo spacing
)

# create freshwater bars below last seqlogo (axes[3])
data_f = top_200_features_melted_for_bars.loc[top_200_features_melted_for_bars['ecosystem_subtype'] == 'proportion_lake']
data_f["position_x"] = data_f["position_x"].astype(float)

axes[3].bar(
    data_f["position_x"], 
    data_f["feature_importance_vals"], 
    color="#2B8CBF",
    width=2
)
axes[3].invert_yaxis()

for ax in axes:
    ax.set_xticks([0, 500])

# Ensure Logomaker uses the same ticks
AA_logo_marine.ax.set_xticks(position_df_new_freshwater.index)
AA_logo_fresh.ax.set_xticks(position_df_new_freshwater.index)

axes[-1].set_xticklabels(position_df_new_freshwater.index, rotation=90, fontsize=100)


# axes[0].set_ylabel("Feature Importance Marine",fontsize=50)
# axes[1].set_ylabel("SeqLogo Marine",fontsize=50)
# axes[2].set_ylabel("SeqLogo Freshwater",fontsize=50)
# axes[3].set_ylabel("Feature Importance Freshwater",fontsize=50)

axes[0].tick_params(labelsize=15)
axes[1].tick_params(labelsize=15)
axes[2].tick_params(labelsize=15)
axes[3].tick_params(labelsize=15)

# plt.yticks(fontsize=80)

# Adjust layout
plt.tight_layout()
# plt.subplots(layout="constrained")

# Display the combined figure
plt.show()

In [ ]:
top_200_features_melted['value'] = abs(top_200_features_melted['value'])

# top_200_features_melted.loc[top_200_features_melted['value']]
temp = top_200_features_melted.sort_values(['variable','value'], ascending=[True,False]).drop_duplicates(['variable'])
temp

grouped = temp.groupby(['ecosystem_subtype','AA']).count().reset_index().drop(['value','position_x'],axis=1)
grouped.rename(columns={'variable' : 'count_of_AA'}, inplace=True)
grouped.sort_values(['ecosystem_subtype','count_of_AA'],ascending=[False, False], inplace=True)
grouped

grouped['count_of_AA_proportion'] = np.where(grouped['ecosystem_subtype'] == "proportion_oceanic", 
                                             grouped['count_of_AA']/61*100,
                                             grouped['count_of_AA']/791*100)
grouped

In [ ]:
sns.set_theme(style="whitegrid")
my_palette = sns.diverging_palette(h_neg=100, h_pos=200,s=100,l=60, n=2)
min,max = 2, 50
plt.figure(figsize=(6,x/50))

b = sns.barplot(
    data=grouped,
    x="count_of_AA",
    y="AA",
    hue="ecosystem_subtype",
    palette=['#9dcc7e','#2B8CBF'],
    dodge=True,
    width=-.5,  
)

# show the graph
# b.set_yticklabels(top_200_features_melted.variable)
b.set_xlabel(f"Count in {x} features",fontsize=15)
b.set_ylabel("Amino Acids",fontsize=15)
b.plot(10, 0, "*", markersize=10, color="C1")
b.plot(10, 1, "*", markersize=10, color="C1")
b.plot(10, 2, "*", markersize=10, color="C1")
b.plot(10, 3, "*", markersize=10, color="C1")
b.plot(10, 10, "*", markersize=10, color="C1")

plt.show()

In [ ]:
# from Bio.SeqUtils.ProtParam import ProteinAnalysis
# from Bio import SeqIO
# from collections import defaultdict

# count_amino_acids_dict = defaultdict(dict)
# for record in SeqIO.parse("training_set_micro_lake_oceanic.faa", "fasta"):
#     X = ProteinAnalysis(record.seq)
#     count_amino_acids_dict[record.id] = X.count_amino_acids()
    
# print(count_amino_acids_dict)

# df_counts_aa = pd.DataFrame.from_dict(count_amino_acids_dict)
# # print(df_counts_aa.sum())
# df_counts_aa_length_norm = (df_counts_aa/df_counts_aa.sum(axis=0)).T.reset_index(names='protein')
# df_counts_aa_length_norm

# # df_counts_aa_length_norm['img_split'] = 

# df_counts_aa_length_norm.loc[df_counts_aa_length_norm['protein'].str.contains('IMG'), 'img_split'] = (df_counts_aa_length_norm.loc[df_counts_aa_length_norm['protein'].str.contains('IMG'), 'protein']
#       .str.split('|').str[0])
# df_counts_aa_length_norm['img_split'] = df_counts_aa_length_norm['img_split'].fillna(df_counts_aa_length_norm['protein'])
# df_counts_aa_length_norm.set_index('img_split',inplace=True)
# df_counts_aa_length_norm.drop(columns='protein', inplace=True)

# biome_info = data_df_merged_balanced[['img_split','ecosystem_subtype']].set_index('img_split')
# df_with_biome = df_counts_aa_length_norm.join(biome_info)
# df_counts_melted = pd.melt(df_with_biome,id_vars='ecosystem_subtype')
# df_counts_melted_grouped = df_counts_melted.groupby(['ecosystem_subtype','variable']).sum() / df_counts_melted.groupby(['ecosystem_subtype','variable']).count()
# df_counts_melted_grouped.reset_index(inplace=True)
# df_counts_melted_grouped.sort_values(['ecosystem_subtype','value'],ascending=[False,False],inplace=True)
# df_counts_melted_grouped.head(n=10)

In [ ]:
# sns.set_theme(style="whitegrid")
# my_palette = sns.diverging_palette(h_neg=100, h_pos=200,s=100,l=60, n=2)
# min,max = 2, 50
# plt.figure(figsize=(6,x/750))

# b = sns.barplot(
#     data=df_counts_melted_grouped,
#     x="value",
#     y="variable",
#     hue="ecosystem_subtype",
#     palette=['#9dcc7e','#2B8CBF'],
#     dodge=True,
#     width=-.5,  
# )

# # show the graph
# # b.set_yticklabels(top_200_features_melted.variable)
# b.set_xlabel("Amino acid",fontsize=15)
# b.set_ylabel("count_of_amino acids in by biome",fontsize=15)
# # b.plot(30, 0, "*", markersize=10, color="C1")
# # b.plot(30, 1, "*", markersize=10, color="C1")
# # b.plot(30, 2, "*", markersize=10, color="C1")
# # b.plot(30, 10, "*", markersize=10, color="C1")

# plt.show()